## **Introduction to Agent Memory**

Memory Types:
1. Short-term: Thread/Session level
2. Long-term: Across conversation

Memory will persist the state between the call to the agent. Memory is especially important for agents that:
- engage in long runing conversations which might be interrupted/paused
- remembers previous conversations so that you can have productive interactions



### **Memory in LangChain Agents**

LangChain's **create_agent()** runs on **LangGraph's runtime** under the hood and has a built-in memory.

- LangChain’s agent **manages short-term memory by default** as a part of your agent’s **messages state**.
- By storing these in the graph’s state, the agent can access the full context for a given conversation while maintaining separation between different threads. **State is persisted** to a database (or memory) **using a checkpointer** so the thread can be resumed at any time.
- Short-term memory updates when the agent is invoked or a step (like a tool call) is completed, and the state is read at the start of each step.
- Long-term memory can be persisted using **store**.

### **LangChain Agent Memory Type**

**Static runtime context (context_schema)** represents immutable data like user metadata, tools, and database connections that are passed to an application at the start of a run via the context argument to invoke/stream. This data does not change during execution.

**Dynamic runtime context (checkpointer for persistence and state_schema for custom schema)** represents mutable data that can evolve during a single run and is managed through the LangGraph state object. This includes conversation history, intermediate results, and values derived from tools or LLM outputs. In LangGraph, the state object acts as short-term memory during a run.

**Dynamic cross-conversation context (store)** represents persistent, mutable data that spans across multiple conversations or sessions and is managed through the LangGraph store. This includes user profiles, preferences, and historical interactions. The LangGraph store acts as long-term memory across multiple runs. This can be used to read or update persistent facts (e.g., user profiles, preferences, prior interactions).

### **create_agent() function signature**

LangChain agent created with **create_agent()** has a built-in memory under the hood via LangGraph. 

```
create_agent( 
    checkpointer: Checkpointer | None = None,
    store: BaseStore | None = None
    context_schema: type[ContextT] | None = None,
    state_schema: type[AgentState[ResponseT]] | None = None,
)
```
- **checkpointer**: Used for persisting the state of the graph (e.g., as chat memory) for a single thread (e.g., a single conversation).
- **context_schema**: Used for injecting Static Information or Dependencies for an agent invocation like user id, db connections, etc... 
- **state_schema**: Custom state schemas must extend AgentState as a TypedDict. When provided, this schema is used instead of AgentState as the base schema for merging with middleware state schemas. This allows users to add custom state fields without needing to create custom middleware.
- **store**: Used for persisting data across multiple threads (e.g., multiple conversations / users).

As of langchain v1.x, **custom state_schemas must be TypedDict types**. Pydantic models and dataclasses are no longer supported.

Defining custom state via middleware is preferred over defining it via state_schema on create_agent because it allows you to keep state extensions conceptually scoped to the relevant middleware and tools.

**state_schema** is still supported for backwards compatibility on create_agent.

Note that LangGraph exposes a Runtime object with the following information:
1. **State:** AgentState or state_schema containing list of messages and other custom details. Thread Memory. AKA **short-term memory**.
2. **Context:** Allows **dependency injection during an agent invocation** like user id, db connections, etc... (i.e. Static Information)
3. **Store:** A `BaseStore` instance used for  persisting **long-term memory**.
4. **Checkpointer:** Used to persist **short-term memory**.


## **Understanding `ToolRuntime`**

**Tools operate inside a managed runtime with shared memory and context boundaries**

LangChain’s `create_agent` runs on LangGraph’s runtime under the hood. This runtime can be accessed inside the tool using `ToolRuntime`. 

Understand that, when a tool runs, it doesn’t run in isolation. It requires:
- current request info
- agent state
- metadata
- execution context

`ToolRuntime` provides this to a tool.

### **Understanding how state and context interact during execution**

You can access runtime information in tools, as well as via custom agent middleware.


```
** Input: "Schedule a meeting with John tomorrow"

** Tool 1: Extract Info
runtime.state["event"] = {
    "name": "Meeting with John",
    "date": "tomorrow"
}

** Tool 2: Create Meeting
event = runtime.state["event"]
user_id = runtime.context["user_id"]
create_meeting_api(user_id, event)
```

In [2]:
# Step 1: Init a chat model
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY, 
    model="gpt-4o-mini", 
    temperature=1
)

In [14]:
# Step 2: Define a Tool which returns the agent state
from langchain.tools import tool, ToolRuntime

@tool
def get_user_info(
    runtime: ToolRuntime
) -> str:
    """Look up user preferences."""
    print(runtime.__dict__.keys())
    print()
    return f"User preferences: Dark Theme"

In [15]:
# Step 3: Pass the State in create_agent using state_schema arg
from langchain.agents import create_agent

agent = create_agent(
    model=openai_chat_model,
    tools=[get_user_info],
)

In [16]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hi! My name is Bob. Can you provide my preferences?")],
        "user_id": "user_123",
        "theme_preference": "dark"
    }, 
    {
        "configurable" : {"thread_id" : "1"}
    }
)

for msg in response["messages"]:
    msg.pretty_print()

dict_keys(['state', 'context', 'config', 'stream_writer', 'tool_call_id', 'store'])

================================ Human Message =================================

Hi! My name is Bob. Can you provide my preferences?
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_zmyb1iGgTsy3ozqozWhkWQVd)
 Call ID: call_zmyb1iGgTsy3ozqozWhkWQVd
  Args:
================================= Tool Message =================================
Name: get_user_info

User preferences: Dark Theme
================================== Ai Message ==================================

Hi Bob! Your preference is set to a dark theme. If you have any other questions or need assistance, feel free to ask!


## **Understanding the `ToolRuntime` Object**

ToolRuntime contains the following:
1. **runtime.state**
    - By default contains "messages" key i.e. `runtime.state["messages"]`
    - You can use this to access the **custom state schema as well** eg: `runtime.state["event"]`
    - State evolves during execution
    - It is mutable memory shared across the agent execution across all the tools
    - You can **persist** the state using a **checkpointer**
    - **Analogy:** Think of this as RAM
2. **runtime.context**
    - Static, external information about the current request
    - You should not update it
    - It comes from outside the tool and remains same across all tools in that request, for eg: user_id, session_id, db_connection_url, etc...
    - Context should NOT be used to store intermediate results
    - **Analogy:** Think of this as RAM
3. **runtime.store**: long-term memory
4. **runtime.config**: configuration like tags, metadata, configurable.thread_id, etc...
5. **runtime.stream_writer**: to share the tool progress
6. **runtime.tool_call_id**: unique id assigned to each tool call


**Note:**
- Context: Immutable and can have a default value
- State: Mutable and can't have a default value

In [3]:
# ! pip install langmem

In [4]:
import langmem